# 13-1. DFIR 자동화 과제와 분석 범위 예제

## Goal

- 자동화 입력·출력·제외 범위를 명시합니다.
- 실제 증거와 합성 학습 자료를 구분합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

경로 문자열과 범위 정의만 다루며 파일을 읽거나 쓰지 않습니다.


## Steps

### 분석 범위 계약 만들기

허용한 아티팩트와 최대 행 수를 변경 불가능한 설정으로 표현합니다.


In [1]:
from dataclasses import dataclass


@dataclass(frozen=True)
class AnalysisScope:
    case_id: str
    allowed_artifacts: frozenset[str]
    max_rows: int
    synthetic: bool


def validate_scope(scope: AnalysisScope) -> AnalysisScope:
    if not scope.case_id.strip():
        raise ValueError("case_id가 필요합니다")
    supported = {"evtx", "prefetch", "registry", "mft", "shimcache", "amcache"}
    if not scope.allowed_artifacts or not scope.allowed_artifacts <= supported:
        raise ValueError("지원하지 않는 아티팩트가 포함되어 있습니다")
    if not 1 <= scope.max_rows <= 100_000:
        raise ValueError("행 수 제한이 범위를 벗어났습니다")
    return scope


scope = validate_scope(AnalysisScope(
    case_id="SYNTHETIC-CASE-001",
    allowed_artifacts=frozenset({"evtx", "prefetch", "registry"}),
    max_rows=10_000,
    synthetic=True,
))
print(scope)


AnalysisScope(case_id='SYNTHETIC-CASE-001', allowed_artifacts=frozenset({'evtx', 'registry', 'prefetch'}), max_rows=10000, synthetic=True)


## Checks

범위가 비어 있거나 지원하지 않는 아티팩트를 포함하면 거부합니다.


In [2]:
assert scope.synthetic is True
try:
    validate_scope(AnalysisScope("CASE", frozenset({"memory"}), 100, False))
except ValueError:
    print("범위 밖 아티팩트 거부 확인")


범위 밖 아티팩트 거부 확인


## Next Steps

실제 사건에서는 승인 범위·수집 원본·작업 사본·결과 저장 위치를 별도 기록합니다.
